# Initial cohort selection
This notebook performs the initial cohort selection for depression subtyping. It filters the dataset based on predefined inclusion and exclusion criteria to create a refined cohort for further study. The steps include loading the dataset, applying filters, and saving the resulting cohorts (together with depression subjects' IDs with suffixes for neuroimaging data extraction, see `02_workstation_pull.ipynb`) for subsequent analysis. Important to note that this is a preliminary step and further refinement may be necessary based on additional criteria or data quality checks (see subsequent notebooks such as `01_cohort_matching.ipynb` and `05_aggregate_dMRI.ipynb`).

## 1) Imports

In [ ]:
import os
import sys
from pprint import pprint

# Make sure we can import cohort_selection_utils the same way as the script
PROJECT_ROOT = '.../subtyping_depression'
COHORT_UTILS_DIR = os.path.join(PROJECT_ROOT, 'source_code', 'cohort_definition')
if COHORT_UTILS_DIR not in sys.path:
    sys.path.insert(0, COHORT_UTILS_DIR)

from cohort_selection_utils import (
    load_ukb_datasets,
    standardize_eid_columns,
    create_cohort,
    extract_subject_ids,
    count_codes_in_cohort,
    build_comorbidity_indicator_matrix,
    plot_comorbidity_distribution,
    save_cohort,
)

%matplotlib inline

## 2) Configuration
Edit this cell to change:
- Input/output paths
- ICD-10 inclusion/exclusion codes
- Which datasets must overlap (`REQUIRED_DATASETS`) vs optional left-merges

### Control definitions
- If `CONTROL_CLEAN = False`: controls exclude only the specified depression codes (e.g., F32/F33), but may still have other ICD-10 diagnoses.
- If `CONTROL_CLEAN = True`: controls are participants with **no ICD-10 codes at all** (according to `create_cohort(..., control_no_icd10=True)`).

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'UKB')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'UKB', 'cohorts')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Control cohort behaviour
CONTROL_CLEAN = True
CONTROL_EXCLUDE_CODES = ['F32', 'F33']  # used only if CONTROL_CLEAN=False

# Depression definition
DEPRESSION_INC = ['F32']
DEPRESSION_EXC = ['F33']
DEPRESSION_CODES = '_'.join(DEPRESSION_INC)

# Suffix to add to extracted subject IDs (for neuroimaging data extraction)
SUBJECT_ID_SUFFIX = '/i2'  # e.g., '/i2' if IDs need to match specific format

# Output files
DEPRESSION_OUTPUT = os.path.join(OUTPUT_DIR, f'depression_cohort_{DEPRESSION_CODES}.csv')
CONTROL_OUTPUT = os.path.join(OUTPUT_DIR, 'control_cohort.csv')

# Datasets: required overlap vs optional merges
REQUIRED_DATASETS = ['cognitive', 'restingfmri', 'dMRI', 'demographics']
OPTIONAL_DATASETS = []

# Plot + coding file
CODING_FILEPATH = os.path.join(DATA_DIR, 'coding19.tsv')
COMORBIDITY_TABLE_OUT = os.path.join(OUTPUT_DIR, 'icd10_code_distribution_depression_cohort.csv')
COMORBIDITY_PLOT_OUT = os.path.join(OUTPUT_DIR, 'comorbidity_distribution_depression_cohort.svg')

## 3) Load datasets
This loads the UKB tables used in cohort creation. You can restrict columns to keep memory usage lower.

In [ ]:
columns_cognitive = [
    'eid', 'p399_i2_a1', 'p399_i2_a2', 'p4282_i2', 'p20018_i2',
    'p20197_i2', 'p23324_i2', 'p6348_i2', 'p20023_i2',
    'p21004_i2', 'p6350_i2', 'p20016_i2', 'p6373_i2', 'p26302_i2',
]
columns_demographics = ['eid', 'p31', 'p21003_i2']

datasets = load_ukb_datasets(
    data_dir=DATA_DIR,
    columns_cognitive=columns_cognitive,
    columns_demographics=columns_demographics,
    convert_sex=True,
)
pprint('Loaded keys:')
pprint(list(datasets.keys()))

## 4) Standardize `eid` columns
Ensures `eid` is consistent (string) across all loaded DataFrames before merging.

In [ ]:
datasets = standardize_eid_columns(datasets, target_type='str')
print('Dataset sizes:')
for name, df in datasets.items():
    try:
        print(f' - {name}: {len(df)}')
    except Exception:
        pass

## 5) Create the depression cohort and extract subject IDs
Includes participants with ICD-10 codes in `DEPRESSION_INC`, and excludes any in `DEPRESSION_EXC`.The cohort is restricted to participants who are present in all `REQUIRED_DATASETS`. Also extracts subject IDs (with suffix '/i2') from the depression cohort for later extraction of neuroimaging data (02_workstation_pull.ipynb).

In [ ]:
depression_cohort = create_cohort(
    diagnosis_df_icd10=datasets['diagnosis_ICD10'],
    cognitive_df=datasets.get('cognitive'),
    taskfmri_df=datasets.get('taskfmri'),
    restingfmri_df=datasets.get('restingfmri'),
    dmri_df=datasets.get('dMRI'),
    demographics_df=datasets.get('demographics'),
    icd10_codes=DEPRESSION_INC,
    cohort_type='depression',
    exclude_icd10_codes=DEPRESSION_EXC,
    required_datasets=REQUIRED_DATASETS,
    optional_datasets=OPTIONAL_DATASETS,
)
display(depression_cohort.head())

extract_subject_ids(
    cohort_df=depression_cohort,
    eid_column='eid',
    suffix=SUBJECT_ID_SUFFIX,
    out_path=os.path.join(OUTPUT_DIR, 'depression_subject_ids.txt'),
)

## 6) Create the control cohort
Two options:
- `CONTROL_CLEAN=True`: controls are those with **no ICD-10 codes at all** (`control_no_icd10=True`).
- `CONTROL_CLEAN=False`: controls exclude only depression codes (`CONTROL_EXCLUDE_CODES`) but may have other diagnoses.

In [ ]:
control_cohort = create_cohort(
    diagnosis_df_icd10=datasets['diagnosis_ICD10'],
    cognitive_df=datasets.get('cognitive'),
    taskfmri_df=datasets.get('taskfmri'),
    restingfmri_df=datasets.get('restingfmri'),
    dmri_df=datasets.get('dMRI'),
    demographics_df=datasets.get('demographics'),
    cohort_type='control',
    icd10_codes=None if CONTROL_CLEAN else CONTROL_EXCLUDE_CODES,
    exclude_icd10_codes=None,
    required_datasets=REQUIRED_DATASETS,
    optional_datasets=OPTIONAL_DATASETS,
    control_no_icd10=CONTROL_CLEAN,
)

display(control_cohort.head())

## 7) Comorbidity distribution (depression cohort)
`count_codes_in_cohort` counts ICD-10 codes across subjects in the depression cohort and (optionally) links codes to textual meanings using `coding19.tsv`.

In [ ]:
comorbidity_distribution = count_codes_in_cohort(
    cohort_df=depression_cohort,
    codes_column_cohort='codes',
    coding_filepath=CODING_FILEPATH,
    output_path=COMORBIDITY_TABLE_OUT,
)
print('Unique ICD-10 codes in depression cohort:', len(comorbidity_distribution))
display(comorbidity_distribution.head(25))

Generate and save a dataframe with for each subject in the depression cohort whether they have each of the top N comorbid ICD-10 codes. This will be used in later analyses to explore comorbidity patterns within depression subtypes and control for most prevalent comorbidity effects. 

In [ ]:
depression_cohort = build_comorbidity_indicator_matrix(cohort_df = depression_cohort,
                                   eid_column = 'eid',
                                   codes_column_cohort = 'codes',
                                   proportion_threshold = 0.10,
                                   exclude_codes = DEPRESSION_INC)
print(depression_cohort.columns)

## 8) Plot comorbidity distribution
The plotting helper supports wrapped y-labels, slight rotation, and additional spacing to reduce overlap for long titles.

In [ ]:
plot_comorbidity_distribution(
    code_distribution=comorbidity_distribution,
    proportion_threshold=0.10,
    title='ICD-10 code distribution in depression cohort (≥10% prevalence)',
    figsize=(20, 20),
    wrap_width=40,
    y_label_rotation=15.0,
    y_spacing=3.0,
    output_path=COMORBIDITY_PLOT_OUT,
)
print('Saved plot to:', COMORBIDITY_PLOT_OUT)

## 9) Save cohorts
Writes both cohorts to CSV for downstream steps (matching, modeling, plotting, etc.).

In [ ]:
save_cohort(depression_cohort, DEPRESSION_OUTPUT)
save_cohort(control_cohort, CONTROL_OUTPUT)

## 10) Summary
A quick report of cohort sizes and some column sanity checks.

In [ ]:
print('Depression cohort:', len(depression_cohort))
print('Control cohort:', len(control_cohort))

print('Depression columns:')
pprint(list(depression_cohort.columns))

print('Extracted depression subject IDs to:', os.path.join(OUTPUT_DIR, 'depression_subject_ids.txt'))

print('Control columns:')
pprint(list(control_cohort.columns))